# On-Disk Transductive Learning: Introduction

Learn the **foundations** of mini-batch training on large graphs with topological learning!

**What you'll learn:**
- ???? What is transductive learning?
- ???? Why mini-batches for large graphs?
- ???? What are node samplers?
- ??????? Cluster-aware sampling basics
- ???? The structure loss problem
- ??????? On-disk indexing solution

**Prerequisites:** Basic PyTorch Geometric and TopoBench familiarity

**Next:** Tutorial 2 (Structure-Centric) or Tutorial 3 (Extended Context)

## Table of Contents

1. [Transductive Learning Basics](#transductive)
2. [The Memory Challenge](#memory)
3. [Node Samplers Explained](#samplers)
4. [Cluster-Aware Sampling](#clusters)
5. [The Structure Loss Problem](#loss)
6. [On-Disk Indexing Solution](#ondisk)
7. [Next Steps](#next)

<a id='transductive'></a>
## 1. Transductive Learning Basics

**Transductive learning** = Single large graph with train/val/test **masks**

- All nodes present from start
- Predict labels for test nodes using full graph structure
- Common in: social networks, citation networks, knowledge graphs

**vs Inductive:** Multiple separate graphs (predict on new graphs)

In [ ]:
import torch
from torch_geometric.data import Data

# Example: Transductive data
num_nodes = 100
edge_index = torch.randint(0, num_nodes, (2, 300))
x = torch.randn(num_nodes, 16)
y = torch.randint(0, 4, (num_nodes,))

# Masks (not separate graphs!)
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[:60] = True
val_mask[60:80] = True
test_mask[80:] = True

data = Data(x=x, edge_index=edge_index, y=y, num_nodes=num_nodes,
            train_mask=train_mask, val_mask=val_mask, test_mask=test_mask)

print(f'??? Single graph: {num_nodes} nodes')
print(f'  Train: {train_mask.sum().item()} | Val: {val_mask.sum().item()} | Test: {test_mask.sum().item()}')
print('  All nodes present, split by masks!')

<a id='memory'></a>
## 2. The Memory Challenge

**Problem:** Large graphs with topological structures are **memory-intensive**!

**Example:**
- 100K nodes, degree 30 ??? ~500K triangles
- Enumerate all in RAM: 1-10 GB+
- Citation/social networks: Even larger!

**Solution:** Mini-batch training with **constant memory**

In [ ]:
# Memory scaling
print('???? Estimated memory for full-graph storage:')
for n in [10_000, 50_000, 100_000]:
    triangles_est = n * 10  # Rough estimate
    mem_mb = (triangles_est * 3 * 8 + n * 128 * 4) / (1024**2)
    print(f'  {n:,} nodes ??? ~{mem_mb:.0f} MB')

print('\n  ?????? Need mini-batches for large graphs!')

<a id='samplers'></a>
## 3. Node Samplers Explained

**What is a node sampler?**

An object that yields **batches of node IDs** when iterated:

```python
for node_batch in sampler:
    # node_batch: torch.Tensor of node IDs
    # Extract subgraph, train on it
```

**Types:**
1. **Random:** Sample N random nodes
2. **Cluster-aware:** Sample nodes from same community (better!)
3. **Layer-wise:** Based on GNN layers

In [ ]:
# Simple random sampler example
class SimpleRandomSampler:
    def __init__(self, num_nodes, batch_size, num_batches):
        self.num_nodes = num_nodes
        self.batch_size = batch_size
        self.num_batches = num_batches
    
    def __iter__(self):
        for _ in range(self.num_batches):
            yield torch.randperm(self.num_nodes)[:self.batch_size]
    
    def __len__(self):
        return self.num_batches

sampler = SimpleRandomSampler(num_nodes=100, batch_size=20, num_batches=5)

print('???? Random Node Sampler:')
for i, nodes in enumerate(sampler):
    print(f'  Batch {i+1}: {len(nodes)} nodes')

print('\n  Each batch: random subset of 20 nodes')

<a id='clusters'></a>
## 4. Cluster-Aware Sampling

**Why clusters?**

Random sampling ??? sparse subgraphs
Cluster sampling ??? **dense subgraphs** (more useful!)

**Community detection methods:**
- **Louvain:** Fast, optimizes modularity
- **METIS:** Graph partitioning, minimizes edge cuts
- **Leiden:** Improved Louvain
- **Label Propagation:** Simple and fast

**TopoBench provides `ClusterAwareNodeSampler`!**

In [ ]:
from topobench.dataloader import ClusterAwareNodeSampler

# Create sample data
sample_data = Data(
    x=torch.randn(100, 16),
    edge_index=torch.randint(0, 100, (2, 300)),
    num_nodes=100
)

# Cluster-aware sampler
sampler = ClusterAwareNodeSampler(
    data=sample_data,
    method='louvain',      # or 'metis', 'leiden'
    nodes_per_batch=20,
    shuffle=True
)

print('??? ClusterAwareNodeSampler created')
print(f'  Method: Louvain')
print(f'  Batches: {len(sampler)}')
print('\n  Samples nodes from same community ??? dense subgraph!')

<a id='loss'></a>
## 5. The Structure Loss Problem

**Challenge:** When extracting subgraphs, structures may be **incomplete** (fragmented)!

### What is Structure Fragmentation?

Imagine your graph is the **1-skeleton** of a triangulated surface (like a mesh):

**❌ Node-First Sampling (Traditional):**
```
1. Pick random nodes: {v1, v2, v3, v5, v8}
2. Extract induced subgraph
3. Check triangles:
   - Triangle (v1, v2, v3): ✓ All nodes present
   - Triangle (v2, v5, v9): ✗ BROKEN! Missing v9
   - Triangle (v8, v10, v11): ✗ BROKEN! Missing v10, v11
```

**Result:** A **torn surface** with incomplete triangular faces - like frayed cloth with loose threads!

**✅ Structure-First Sampling (Our Solution):**
```
1. Pick complete triangles: {T1, T2, T3}
2. Gather ALL their nodes automatically
3. Every triangle guaranteed complete!
```

**Result:** **Intact surface patches** - all topological information preserved!

### Visual Example

```
Full Graph:     Node-First:    Structure-First:
  5 --- 47        5              5 --- 47
   \   /          \               \   /
    \ /            \               \ /
     12            12              12
Triangle ✓      Broken ✗        Complete ✓
```

**Typical loss with node-first:** 20-40% of structures at cluster boundaries!

**Why this matters:** For topological deep learning, you need **intact structures** to learn curvature, homology, and higher-order patterns!

<a id='ondisk'></a>
## 6. On-Disk Indexing Solution

**Key insight:** Index all structures once, query on-demand!

### The On-Disk Index

**Build once:**
```
Graph ??? Enumerate Structures ??? Stream to SQLite ??? Store on Disk
```

**Query many times:**
```
Query: 'Structures with nodes [5, 12, ...]'
  ???
SQLite ??? Fast Lookup ??? Return Complete Structures
```

**Benefits:**
- ??? Constant memory (only batch in RAM)
- ??? Persistent (build once, reuse forever)
- ??? Fast queries (<1ms)
- ??? Enables structure-complete batching!

### Memory Comparison

| Approach | Memory | Storage |
|----------|--------|--------|
| In-Memory | 5-30 GB RAM | N/A |
| On-Disk | ~200 MB RAM | 50-200 MB disk |

In [ ]:
from topobench.data.preprocessor import OnDiskTransductivePreprocessor
from omegaconf import OmegaConf

# Configure transform
transforms_config = OmegaConf.create({
    'clique_lifting': {
        'transform_type': 'lifting',
        'transform_name': 'SimplicialCliqueLifting',
        'complex_dim': 2
    }
})

# Build on-disk index
preprocessor = OnDiskTransductivePreprocessor(
    graph_data=sample_data,
    data_dir='./index_demo',
    transforms_config=transforms_config,
    max_structure_size=3
)

print('??????? Building on-disk index...')
preprocessor.build_index()

print(f'\n??? Index built!')
print(f'  Structures: {preprocessor.num_structures:,}')
print(f'  Location: ./index_demo/')
print('\n  ???? Index is persistent - build once, reuse forever!')

<a id='next'></a>
## 7. Next Steps

You now understand the foundations! Choose your path:

### Tutorial 2: Structure-Centric Batching
**→ `tutorial_ondisk_transductive_structure_centric.ipynb`**

- Sample structures first, then gather nodes
- **Prevents fragmentation** - all nodes of each structure included
- **100% structure completeness** by construction
- Best for: Guaranteed completeness, topological liftings (simplicial, cell, hypergraph)

### Tutorial 3: Extended Context Batching  
**→ `tutorial_ondisk_transductive_extended_context.ipynb`**

- Enhance existing node samplers with context expansion
- **Repairs fragmentation** by adding missing nodes
- **95-100% completeness** with controlled expansion
- Best for: Backward compatibility, existing samplers

### Quick Comparison

| Feature | Structure-Centric | Extended Context |
|---------|------------------|------------------|
| Sampling | Structures first | Nodes first |
| Fragmentation | Prevented | Repaired |
| Completeness | 100% | 95-100% |
| Use existing samplers | No | Yes |
| Memory | ~200 MB | ~300 MB |
| Best for | Guaranteed intact structures | Enhancing samplers |

## 💡 Advanced Note: Why Not Lazy Graph Loading?

**Q: If we're doing "on-disk" processing, shouldn't we lazy-load the graph itself?**

**A: No! The graph fits in RAM - it's the structures that don't.**

### Memory Breakdown (100K node graph)

```
Graph (nodes, edges, features):  ~50-200 MB  ✅ Fits in RAM
Topological structures:           1-10 GB    ⚠️ Needs disk
```

### Why Graph Must Be in Memory

1. **Subgraph extraction** needs full `edge_index` accessible
2. **Preprocessor queries** need fast edge lookups
3. **Feature extraction** needs full `x` tensor
4. **Graph is not the bottleneck** - structures are!

### When Would You Need Lazy Graph Loading?

**Only for truly massive graphs:**
- 10M+ nodes (very rare!)
- Billions of edges
- Graph itself doesn't fit in RAM
- Would require custom preprocessing (our tools won't work)

**For 99% of users:** `InMemoryDataset` + on-disk structure index = optimal!

**This is why Tutorials 2 & 3 use `InMemoryDataset`** - it's the correct choice!

## Summary

**What we learned:**
1. ??? Transductive learning: Single graph with masks
2. ??? Memory challenge: Large graphs need mini-batches
3. ??? Node samplers: Yield batches of node IDs
4. ??? Cluster-aware: Sample from communities (dense subgraphs)
5. ??? Structure loss: 20-40% lost at cluster boundaries
6. ??? On-disk indexing: Constant memory, persistent storage

**Key takeaway:** On-disk indexing enables **structure-aware mini-batch training** on large graphs!

**Continue to Tutorial 2 or 3 to learn advanced approaches!** ????

In [ ]:
# Cleanup
preprocessor.close()
print('??? Tutorial complete!')